# Feature Engineering para Dados de Carne Brasileira
Cria uma cópia do dataset original e adiciona features derivadas de séries temporais.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose

# Carregar dataset original sem modificá-lo
PATH = "../carne-brasileira-exportada.csv"
df = pd.read_csv(PATH)

df["date"] = pd.to_datetime(df["ano"].astype(str) + "Q" + df["trimestre"].astype(str))
df = df.sort_values(["tipo_carne", "date"]).reset_index(drop=True)

# Criar cópia com features derivadas

def add_derived_features(group):
    group = group.copy()
    group = group.sort_values("date")
    group["time_index"] = np.arange(len(group))
    group["quarter_sin"] = np.sin(2 * np.pi * (group["trimestre"] - 1) / 4)
    group["quarter_cos"] = np.cos(2 * np.pi * (group["trimestre"] - 1) / 4)
    group["halfyear_sin"] = np.sin(2 * np.pi * (group["trimestre"] - 1) / 2)
    group["halfyear_cos"] = np.cos(2 * np.pi * (group["trimestre"] - 1) / 2)

    for col in ["exportacao_usd", "preco_medio_ton_usd"]:
        for lag in [1, 2, 3, 4]:
            group[f"{col}_lag{lag}"] = group[col].shift(lag)

        group[f"{col}_rolling_mean_4"] = group[col].shift(1).rolling(window=4, min_periods=1).mean()
        group[f"{col}_ewm_4"] = group[col].shift(1).ewm(span=4, adjust=False).mean()
        group[f"{col}_diff_1"] = group[col].diff(1)
        group[f"{col}_pct_change_a"] = group[col].pct_change(periods=4)

        try:
            series = group.set_index("date")[col].asfreq("Q")
            decomposition = seasonal_decompose(series, model="additive", period=4, extrapolate_trend="freq")
            group.loc[series.index, f"{col}_trend"] = decomposition.trend.values
            group.loc[series.index, f"{col}_seasonal"] = decomposition.seasonal.values
            group.loc[series.index, f"{col}_resid"] = decomposition.resid.values
        except Exception:
            group[f"{col}_trend"] = np.nan
            group[f"{col}_seasonal"] = np.nan
            group[f"{col}_resid"] = np.nan

    return group

# Aplicar por tipo de carne para preservar dependências temporais dentro de cada série

df_derived = df.groupby("tipo_carne", group_keys=False).apply(add_derived_features).reset_index(drop=True)

# Salvar a cópia derivada
OUTPUT_PATH = "../carne-brasileira-derivada.csv"
df_derived.to_csv(OUTPUT_PATH, index=False)

# Visualizar as primeiras linhas
print(f"Derived dataset saved to: {OUTPUT_PATH}")
df_derived.head()


Derived dataset saved to: ../carne-brasileira-derivada.csv


/tmp/ipykernel_35462/1551264741.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["ano"].astype(str) + "Q" + df["trimestre"].astype(str))
/tmp/ipykernel_35462/1551264741.py:33: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  series = group.set_index("date")[col].asfreq("Q")
/tmp/ipykernel_35462/1551264741.py:33: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  series = group.set_index("date")[col].asfreq("Q")
/tmp/ipykernel_35462/1551264741.py:33: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  series = group.set_index("date")[col].asfreq("Q")
/tmp/ipykernel_35462/1551264741.py:33: FutureWarning: 'Q' is deprecated and will be removed in a future version, please

,ano,trimestre,tipo_carne,producao_cabecas_anual_mi,abate_cabecas,peso_carcaca_ton,exportacao_usd,media_cambio_usdbrl,custo_milho_rs,custo_soja_rs,...,preco_medio_ton_usd_lag2,preco_medio_ton_usd_lag3,preco_medio_ton_usd_lag4,preco_medio_ton_usd_rolling_mean_4,preco_medio_ton_usd_ewm_4,preco_medio_ton_usd_diff_1,preco_medio_ton_usd_pct_change_a,preco_medio_ton_usd_trend,preco_medio_ton_usd_seasonal,preco_medio_ton_usd_resid
0,2014,1,bovino,212.34,8.366,1.951,1.345,2.34,22.57,62.01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014,2,bovino,212.34,8.516,2.006,1.384,2.22,23.59,61.42,...,NaN,NaN,NaN,4.403000,4.403000,0.326,NaN,NaN,NaN,NaN
2,2014,3,bovino,212.34,8.456,2.036,1.547,2.28,19.97,56.50,...,4.403,NaN,NaN,4.566000,4.533400,0.145,NaN,NaN,NaN,NaN
3,2014,4,bovino,212.34,8.525,2.058,1.519,2.59,21.46,56.73,...,4.729,4.403,NaN,4.668667,4.669640,-0.016,NaN,NaN,NaN,NaN
4,2015,1,bovino,215.20,7.737,1.836,0.993,2.87,22.95,60.01,...,4.874,4.729,4.403,4.716000,4.744984,-0.579,-0.028163,NaN,NaN,NaN


In [10]:
# Exportar o dataset derivado com as features
OUTPUT_PATH = "../carne-brasileira-derivada.csv"
df_derived.to_csv(OUTPUT_PATH, index=False)
print(f"Derived dataset exported to: {OUTPUT_PATH}")


Derived dataset exported to: ../carne-brasileira-derivada.csv
